In [ ]:
import pandas as pd
import warnings
from sentence_transformers import SentenceTransformer

# Suppress specific NumPy warning
warnings.filterwarnings("ignore", category=UserWarning, message="Failed to initialize NumPy")

# Load model
pubmed_bert = SentenceTransformer("TimKond/S-PubMedBert-MedQuAD")

def cal_semantic_transformer_similarity(s1, s2, model) -> float:
    # Ensure both inputs are strings (convert any type to string)
    s1 = str(s1)
    s2 = str(s2)

    # Encode sentences using the transformer model
    embeds1 = model.encode(s1, convert_to_tensor=True)
    embeds2 = model.encode(s2, convert_to_tensor=True)
    
    # Compute the similarity between the embeddings
    sim_value = model.similarity(embeds1, embeds2)
    
    # Return the similarity score rounded to three decimal places
    return round(float(sim_value[0][0]), 3)

def calculate_similarity_from_excel(input_excel: str, output_excel: str):
    # Load the input Excel file into a pandas DataFrame
    df = pd.read_excel(input_excel)
    
    # Ensure the columns contain sentence pairs, adjust column names as needed
    if 'Sentence1' not in df.columns or 'Sentence2' not in df.columns:
        print("The input file should contain 'Sentence1' and 'Sentence2' columns.")
        return
    
    # Convert any non-string values to strings and handle missing values (NaN)
    df['Sentence1'] = df['Sentence1'].fillna('').apply(str)  # Replace NaN with empty string and convert to string
    df['Sentence2'] = df['Sentence2'].fillna('').apply(str)  # Replace NaN with empty string and convert to string
    
    # Calculate similarity for each pair of sentences and store the results in a new column
    df['Similarity_Score'] = df.apply(lambda row: cal_semantic_transformer_similarity(row['Sentence1'], row['Sentence2'], pubmed_bert), axis=1)
    
    # Save the results to a new Excel file
    df.to_excel(output_excel, index=False)

    print(f"Similarity scores saved to {output_excel}")

# Example usage
input_excel_file = '/Users/kennedytavaris/Desktop/inputsentences3.xlsx'  # Input file with 'Sentence1' and 'Sentence2' columns
output_excel_file = '/Users/kennedytavaris/Desktop/output_similarity_scores4.xlsx'  # Output file to store the similarity scores

calculate_similarity_from_excel(input_excel_file, output_excel_file)
